# Part -1 (class work) : Text Pre-processing in NLP.

# Basics of Text Data Cleaning.
## Instructions and Requirements:

In this Notebook we will evaluate few basic text data cleaning techniques which are most common for any NLP tasks.

This Notebook makes use of "NLTK" and "Regex" Library a lot.

Dataset: "trump_tweet_sentiment_analysis.csv"

This week workshop will have two sections:

To DO:

Do - 1 - Read the code provided, understand their usages and Complete Exercise-1, which is at bottom.

Do - 2 - Based on your implementations Demonstrate the importance of Text pre-processing in NLP (one per group).



## Time to Complete - 90 mins.

The first step in any Natural Language Processing task is to pre-process the text dataset. The main goal of this step is to remove noise from the data. The noise in text data can be in different forms, so in this section we will look into some common data cleaning tasks performed before any NLP task.


 Terminology Alert!!!
*   Document: A distinct unit of text. This could be a sentence, paragraph or an article.

  Example:

1. doc1==> "How are you?"
2. doc2==> "I go to school."
*   Corpus: collection of documents.

Example: corpus=[doc1, doc2]


### Step 0: Install Required Libraries

In [1]:
# Install / import required libraries
import re
import pandas as pd
import numpy as np
import nltk

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Setup stopwords
stop_words = set(stopwords.words('english'))
custom_stopwords = ['@', 'RT']
stop_words.update(custom_stopwords)

print('All libraries imported successfully!')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


All libraries imported successfully!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Read the data.

In [3]:
# NOTE: Update this path to the location of your dataset file
# If using Google Colab with Drive:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/AI_Sem6/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")

data = pd.DataFrame(df)
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [4]:
df_text = data[['text']]   # it takes DataFrame
df_text.dropna()

,text
0,RT @JohnLeguizamo: #trump not draining swamp b...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...
2,Trump protests: LGBTQ rally in New York https:...
3,"""Hi I'm Piers Morgan. David Beckham is awful b..."
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...
...,...
1850118,Everytime im like 'How the fuck I follow Melan...
1850119,RT @imgur: The Trump Handshake. https://t.co/R...
1850120,"""Greenspan warns Trump's policies risk inflati..."
1850121,RT @FasinatingLogic: We must also #INVESTIGATE...


## Removing Unwanted Text.

### Remove URLS:

In this step we will try to remove URLs.

In [5]:
import re
def remove_urls(text):
  """
  This function will try to remove URL present in our dataset and replace it with space using regex library.
  Input Args:
  text: strings of text that may contain URLs.
  Output Args:
  text: URLs replaced with empty string
  """
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)


In the above url_pattern,

**https?://** → matches http or https  

**\S+** → matches non-space characters  

**|** → OR  

**www.\S+** → matches URLs starting with www

In [6]:
text = " Hello, Click on this link to open facebook https://www.facebook.com/"
text_url = remove_urls(text)

In [7]:
text_url

' Hello, Click on this link to open facebook '

In [8]:
text_no_url = df_text["text"].apply(remove_urls)

In [9]:
text_no_url

,text
0,RT @JohnLeguizamo: #trump not draining swamp b...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...
2,Trump protests: LGBTQ rally in New York by #B...
3,"""Hi I'm Piers Morgan. David Beckham is awful b..."
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...
...,...
1850118,Everytime im like 'How the fuck I follow Melan...
1850119,RT @imgur: The Trump Handshake.
1850120,"""Greenspan warns Trump's policies risk inflati..."
1850121,RT @FasinatingLogic: We must also #INVESTIGATE...


### Remove Unwanted Characters.

This may be punctuation, numbers, emoji, dates etc.

[ It depends on dataset and task we are performing. For example, The dataset we are using is scraped from twitter - Thus we will also try to remove @tag and #mentions from the dataset.]



### Remove Emojis:


In [10]:
def remove_emoji(string):
  """
  This function will replace the emoji in string with whitespace
  """
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)

In [11]:
test_string = "Hello @siman 👋🏾, still on up for the movie???  #MovieNight #friday #🍱"
no_emoji = remove_emoji(test_string)
no_emoji

'Hello @siman  , still on up for the movie???  #MovieNight #friday # '

### Remove Every unwanted characters:

We will try to compile everything into one single function to remove everything.

In [12]:
def removeunwanted_characters(document):
  """
  This function will remove all the unwanted characters from the input dataset.
  Input Args:
  document: A text data to be cleaned.
  Return:
  A cleaned document.
  """
  # remove user mentions
  document = re.sub("@[A-Za-z0-9_]+"," ", document)
  # remove hashtags
  document = re.sub("#[A-Za-z0-9_]+","", document)
  # remove punctuation
  document = re.sub("[^0-9A-Za-z ]", "" , document)
  #remove emojis
  document = remove_emoji(document)
  # remove double spaces
  document = document.replace('  ',"")
  return document.strip()       # Removes extra spaces from the start and end of a string.

In [13]:
# Test:
cleaned_string = removeunwanted_characters(test_string)
cleaned_string


'Hellostill on up for the movie'

In [14]:
text_removed_unwanted = df_text["text"].apply(removeunwanted_characters)
text_removed_unwanted

,text
0,RTnot draining swamp but our taxpayer dollars ...
1,ICYMI Hackers Rig FM Radio Stations To Play An...
2,Trump protests LGBTQ rally in New York httpstc...
3,Hi Im Piers Morgan David Beckham is awful but ...
4,RT Tech Firm Suing BuzzFeed for Publishing Unv...
...,...
1850118,Everytime im like How the fuck I follow Melani...
1850119,RT The Trump Handshake httpstcoRI78itAbC4 http...
1850120,Greenspan warns Trumps policies risk inflation...
1850121,RT We must also s who voted NOT to allow the r...


### Tokenizations:

Example:

IN:

"He did not try to navigate after the first bold flight, for the reaction had taken something out of his soul."

OUT:

['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']

We will be using NLTK library to perform tokenizations.


In [15]:
# Test case:
IN = "He did not try to navigate after the first bold flight, for the reaction had taken something out of his soul."
OUT = word_tokenize(IN)
OUT

['He',
 'did',
 'not',
 'try',
 'to',
 'navigate',
 'after',
 'the',
 'first',
 'bold',
 'flight',
 ',',
 'for',
 'the',
 'reaction',
 'had',
 'taken',
 'something',
 'out',
 'of',
 'his',
 'soul',
 '.']

### Remove Punctuations:


In [16]:
def remove_punct(text):
  """
  This function removes the punctuations present in our text data.
  Input Args:
  text: text data.
  Returns:
  text: cleaned text.
  """
  tokenizer = RegexpTokenizer(r"\w+")     #\w = any word character (letters, numbers, underscore) and   + = one or more of them together

  lst=tokenizer.tokenize(' '.join(text))
  return lst


In [17]:
#Test
text_punctutation = "He did not try to navigate: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul."
text_punc_token = word_tokenize(text_punctutation)
print(text_punctutation)
print("+++++++++++++++++_____________________+++++++++++++++++++++")
print(text_punc_token)
print("_____________________+++++++++++++++++++++++++++__________________")
text_clean = remove_punct(text_punc_token)
print(text_clean)

He did not try to navigate: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul.
+++++++++++++++++_____________________+++++++++++++++++++++
['He', 'did', 'not', 'try', 'to', 'navigate', ':', 'after', 'the', '!', '!', '!', '!', 'first', 'bold', 'flight', ',', 'for', ',', ',', ',', ',', ',', 'the', 'reaction', '!', '!', '!', '!', 'had', 'taken', '?', '?', '?', '?', '?', '?', '?', 'something', 'out', 'of', 'his', 'soul', '.']
_____________________+++++++++++++++++++++++++++__________________
['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul']


### Remove StopWord:

A majority of the words in a given text are connecting parts of a sentence rather than showing subjects, objects or intent. Words like "the" or "and" can be removed by comparing text to a list of stopwords provided by the NLTK library.

We can also define stopwords as required by our task and dataset requirement.



In [18]:

def remove_stopwords(text_tokens):
  """
  This function removes all the stopwords present in our text tokens.
  Input Args:
  text_tokens: tokenized input of our datasets.
  Returns:
  result_tokens: list of token without stopword.
  """

  result_tokens = []
  for token in text_tokens:
    if token not in stop_words:
       result_tokens.append(token)
  return result_tokens

In [19]:
test_inputs = ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']
print(test_inputs)
tokens_without_stopwords = remove_stopwords(test_inputs)
print(tokens_without_stopwords)

['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']
['He', 'try', 'navigate', 'first', 'bold', 'flight', ',', 'reaction', 'taken', 'something', 'soul', '.']


## Text Normalization:

This is the idea of reducing number of words present in Corpus by the process of Lemmatization, Stemming, Capital to Lower [i.e. My -- my].


### Lemmatization:

It is a common NLP technique used to reduce number of tokens(words) in dataset, this is achieved by replacing the word with its root words.

Example:


In [20]:
def lemmatization(token_text):
  """
  This function performs the lemmatization operations as explained above.
  Input Args:
  token_text: list of tokens.
  Returns:
  lemmatized_tokens: list of lemmatized tokens.
  """
  lemma_tokens = []
  wordnet = WordNetLemmatizer()
  lemmatized_tokens = [wordnet.lemmatize(token, pos = 'v') for token in token_text]

  return lemmatized_tokens




In [21]:
lemmatization("Should we go walking or swimming".split())

['Should', 'we', 'go', 'walk', 'or', 'swim']

### Stemming:

Also a token(word) reduction technique. This technique tries to reduce by chopping off a part of the word at the tail end.


In [22]:
def stemming(text):
  """
  This function performs stemming operations.
  Input Args:
  token_text: list of tokenize text.
  Returns:
  stemm_tokes: list of stemmed tokens.
  """
  porter = PorterStemmer()
  stemm_tokens = []
  for word in text:
    stemm_tokens.append(porter.stem(word))
  return stemm_tokens

In [23]:
#Test
print("+++++++++++++++++++++++++++++++" "INPUT TOKENS" "++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
token_text_test=['Connects','Connecting','Connections','Connected','Connection','Connectings','Connect']
print(token_text_test)
print("++++++++++++++++++" "LEMMATIZED TOKENS" "+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
lemma_tokens = lemmatization(token_text_test)
print(lemma_tokens)
print("+++++++++++++++++++++" "STEMMED TOKENS" "+++++++++++++++++++++++++++++++++++++")
stemmed_tokens = stemming(token_text_test)
print(stemmed_tokens)


+++++++++++++++++++++++++++++++INPUT TOKENS++++++++++++++++++++++++++++++++++++++++++++++++++++++++
['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connectings', 'Connect']
++++++++++++++++++LEMMATIZED TOKENS+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connectings', 'Connect']
+++++++++++++++++++++STEMMED TOKENS+++++++++++++++++++++++++++++++++++++
['connect', 'connect', 'connect', 'connect', 'connect', 'connect', 'connect']


### Lower order:


In [24]:
def lower_order(text):
  """
  This function converts all the text in input text to lower order.
  Input Args:
  token_text : input text.
  Returns:
  small_order_text : text converted to small/lower order.
  """
  small_order_text = text.lower()
  return small_order_text

# Test:
sample_text = "This Is some Normalized TEXT"
sample_small = lower_order(sample_text)
print(sample_small)


this is some normalized text


## Create Input Text Pipeline

We will compile every basic cleaning step in the following function and implement with our dataset.

### Exercise-1:
Read the provided data "trump_tweet_sentiment_analysis.csv" and complete the following compiling function.


Read data:

In [26]:
# Load dataset - update path as needed
data = pd.read_csv("/content/drive/MyDrive/AI_Sem6/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")
data.head()


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [27]:
data_cleaning = data["text"].dropna()

In [28]:
data_cleaning[0]

'RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrumpÂ\x85 https://t.co/gFBvUkMX9z'

In [29]:
# ============================================================
# EXERCISE-1: COMPLETED text_cleaning_pipeline function
# ============================================================

def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  A complete text cleaning pipeline that performs:
    1. Lowercasing
    2. URL removal
    3. Emoji removal
    4. Removal of unwanted characters (mentions, hashtags, punctuation)
    5. Tokenization (via split)
    6. Stopword removal
    7. Lemmatization OR Stemming based on `rule` argument

  Input Args:
    dataset (str)  : A single text string to be cleaned.
    rule    (str)  : 'lemmatize' (default) or 'stem'

  Returns:
    str: A single cleaned string of space-joined tokens.
  """
  # 1. Convert the input to small/lower order.
  data = lower_order(dataset)

  # 2. Remove URLs
  data = remove_urls(data)

  # 3. Remove emojis
  data = remove_emoji(data)

  # 4. Remove all other unwanted characters (mentions, hashtags, punctuation, etc.)
  data = removeunwanted_characters(data)

  # 5. Create tokens via simple split (whitespace tokenization)
  tokens = data.split()

  # 6. Remove stopwords
  tokens = [word for word in tokens if word not in stop_words]

  # 7. Apply lemmatization or stemming
  wordnet_lemmatizer = WordNetLemmatizer()
  porter_stemmer     = PorterStemmer()

  if rule == "lemmatize":
    tokens = [wordnet_lemmatizer.lemmatize(word, pos='v') for word in tokens]
  elif rule == "stem":
    tokens = [porter_stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)


In [30]:
# Test with a single string
sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱"
print("--- Lemmatize ---")
print(text_cleaning_pipeline(sample))
print("--- Stem ---")
print(text_cleaning_pipeline(sample, rule="stem"))

--- Lemmatize ---
hello still want us hit new sushi spot lmk youre free cuz cant go next weekend since ill swim
--- Stem ---
hello still want us hit new sushi spot lmk your free cuz cant go next weekend sinc ill swim


In [31]:
# Test on first row of dataset
test = data["text"][0]
print(text_cleaning_pipeline(test))

rtnot drain swamp taxpayer dollars trip advertise properties


In [32]:
  # Apply the pipeline to the entire 'text' column
  cleaned_tokens = data["text"].apply(lambda txt: text_cleaning_pipeline(txt))
  cleaned_tokens

,text
0,rtnot drain swamp taxpayer dollars trip advert...
1,icymi hackers rig fm radio station play antitr...
2,trump protest lgbtq rally new yorkbyvia
3,hi im piers morgan david beckham awful donald ...
4,rt tech firm sue buzzfeed publish unverified t...
...,...
1850118,everytime im like fuck follow melania trump re...
1850119,rt trump handshake
1850120,greenspan warn trump policies risk inflation s...
1850121,rt must also vote allow release trump tax


In [33]:
# Inspect the first cleaned entry
cleaned_tokens[0]

'rtnot drain swamp taxpayer dollars trip advertise properties'